In [1]:
%%writefile assignment_7.cu
#include <iostream>
#include <cuda_runtime.h>
#include <device_launch_parameters.h>
#include <stdio.h>
#include <chrono>

#define N 1024
#define MERGE_N 1000

// ==========================================
// PROBLEM 1: DIFFERENT TASKS (Summation)
// ==========================================
__global__ void summationKernel(int n, long long *results) {
    int tid = threadIdx.x;

    // Thread 0 performs iterative sum
    if (tid == 0) {
        long long sum = 0;
        for (int i = 1; i <= n; i++) sum += i;
        results[0] = sum;
    }
    // Thread 1 performs formula sum: n(n+1)/2
    else if (tid == 1) {
        results[1] = (long long)n * (n + 1) / 2;
    }
}

// ==========================================
// PROBLEM 2: PARALLEL MERGE SORT (Bitonic-style / Recursive)
// ==========================================
__device__ void merge(int* res, int left, int mid, int right) {
    int temp[MERGE_N];
    int i = left, j = mid + 1, k = left;
    while (i <= mid && j <= right) {
        if (res[i] <= res[j]) temp[k++] = res[i++];
        else temp[k++] = res[j++];
    }
    while (i <= mid) temp[k++] = res[i++];
    while (j <= right) temp[k++] = res[j++];
    for (i = left; i <= right; i++) res[i] = temp[i];
}

__global__ void gpu_mergesort(int* data, int size, int width) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int start = idx * width * 2;
    if (start < size) {
        int mid = start + width - 1;
        int end = min(start + 2 * width - 1, size - 1);
        if (mid < end) merge(data, start, mid, end);
    }
}

// ==========================================
// PROBLEM 3: VECTOR ADDITION & BANDWIDTH
// ==========================================
#define VEC_SIZE 1000000
__device__ float dev_a[VEC_SIZE];
__device__ float dev_b[VEC_SIZE];
__device__ float dev_c[VEC_SIZE];

__global__ void vectorAddStatic() {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < VEC_SIZE) {
        dev_c[i] = dev_a[i] + dev_b[i];
    }
}

// ==========================================
// MAIN EXECUTION
// ==========================================
int main() {
    // --- PROBLEM 1 EXECUTION ---
    printf("--- Problem 1: Summation ---\n");
    long long *d_results, h_results[2];
    cudaMalloc(&d_results, 2 * sizeof(long long));
    summationKernel<<<1, 2>>>(N, d_results);
    cudaMemcpy(h_results, d_results, 2 * sizeof(long long), cudaMemcpyDeviceToHost);
    printf("Iterative Sum: %lld, Formula Sum: %lld\n\n", h_results[0], h_results[1]);

    // --- PROBLEM 2 EXECUTION (Partial implementation for brevity) ---
    printf("--- Problem 2: Merge Sort ---\n");
    int *h_data = (int*)malloc(MERGE_N * sizeof(int));
    int *d_data;
    for(int i=0; i<MERGE_N; i++) h_data[i] = rand() % 1000;
    cudaMalloc(&d_data, MERGE_N * sizeof(int));
    cudaMemcpy(d_data, h_data, MERGE_N * sizeof(int), cudaMemcpyHostToDevice);

    for (int width = 1; width < MERGE_N; width *= 2) {
        int blocks = (MERGE_N + (2 * width) - 1) / (2 * width);
        gpu_mergesort<<<blocks, 1>>>(d_data, MERGE_N, width);
        cudaDeviceSynchronize();
    }
    printf("Merge Sort completed on GPU.\n\n");

    // --- PROBLEM 3: BANDWIDTH & PROFILING ---
    printf("--- Problem 3: Bandwidth Analysis ---\n");
    float *h_a = (float*)malloc(VEC_SIZE * sizeof(float));
    for(int i=0; i<VEC_SIZE; i++) h_a[i] = 1.0f;

    // Use cudaMemcpyToSymbol for static device variables
    cudaMemcpyToSymbol(dev_a, h_a, VEC_SIZE * sizeof(float));
    cudaMemcpyToSymbol(dev_b, h_a, VEC_SIZE * sizeof(float));

    // Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    vectorAddStatic<<<(VEC_SIZE + 255) / 256, 256>>>();
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    // Bandwidth Calculation
    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);

    // Theoretical Bandwidth
    // clockRate is in kHz, busWidth in bits.
    // Double data rate (2) * clock * busWidth (bits to Bytes / 8)
    double theoreticalBW = 2.0 * prop.memoryClockRate * (prop.memoryBusWidth / 8.0) / 1e6;

    // Measured Bandwidth: (Read A + Read B + Write C) / time
    double bytesRead = VEC_SIZE * sizeof(float) * 2;
    double bytesWritten = VEC_SIZE * sizeof(float);
    double measuredBW = ((bytesRead + bytesWritten) / (milliseconds / 1000.0)) / 1e9;

    printf("Kernel Time: %f ms\n", milliseconds);
    printf("Theoretical Bandwidth: %f GB/s\n", theoreticalBW);
    printf("Measured Bandwidth: %f GB/s\n", measuredBW);

    return 0;
}

Writing assignment_7.cu


In [2]:
!nvcc -o assignment assignment_7.cu
!./assignment

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
--- Problem 1: Summation ---
Iterative Sum: 524800, Formula Sum: 524800

--- Problem 2: Merge Sort ---
Merge Sort completed on GPU.

--- Problem 3: Bandwidth Analysis ---
Kernel Time: 0.056192 ms
Theoretical Bandwidth: 320.064000 GB/s
Measured Bandwidth: 213.553533 GB/s
